# 04 · Judge DPO vs SFT with a Kaggle model (Task 7)
**Create this notebook from Kaggle Benchmarks** (a *utility benchmark* / task notebook) so `kaggle_benchmarks` and the free model proxy are available. **No GPU needed.**

**Needs:** notebook **03**'s output attached. If a previous version of *this* notebook timed out, also attach its output; finished judgments are reused.
**Produces:** `outputs/eval/*_judged.jsonl`, `*.failures.jsonl`, `*.summary.json`, `summary_table.csv` (~0.5 h).

Never regenerates outputs: it only reads the saved pair files.

In [ ]:
import sys, shutil, glob, os, json
from pathlib import Path
# Benchmark notebooks run in their own container, and attached inputs may not be mounted.
# 1) look for notebook 03's output anywhere it might be mounted; 2) else clone the repo,
#    whose results/eval/ folder holds the same pair files.
import subprocess
REPO_URL = "https://github.com/NifraWahaj/cs5323_PA2.git"
roots = [Path(p) for p in ("/kaggle/input", "/kaggle", "/mnt", str(Path.cwd())) if Path(p).exists()]
def scan(pattern):
    for r in roots:
        try:
            hits = sorted(r.rglob(pattern))
        except Exception:
            continue
        if hits:
            return hits
    return []
code = scan("code/sproutlm")
pair_src = scan("*_pairs.jsonl")
if not code or not pair_src:
    print("notebook 03 output not mounted here; cloning repo instead")
    if not Path("sprout_repo").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "sprout_repo"], check=True)
    code = [Path("sprout_repo/sproutlm")]
    pair_src = sorted(Path("sprout_repo/results/eval").glob("*_pairs.jsonl"))
assert pair_src, "no *_pairs.jsonl found: push results/eval/*_pairs.jsonl to the repo"
sys.path.insert(0, str(code[0].parent))
from sproutlm import judge

work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
E = work / "outputs" / "eval"; E.mkdir(parents=True, exist_ok=True)
for f in pair_src:
    shutil.copy(f, E / f.name)
for f in scan("*_judged.jsonl") + scan("*_judged.failures.jsonl"):
    if f.parent != E:
        shutil.copy(f, E / f.name)        # resume from an earlier judging version, if mounted
print("code:", code[0]); print("eval dir:", E); print(sorted(p.name for p in E.iterdir()))

In [ ]:
import kaggle_benchmarks as kbench
JUDGE = "google/gemini-3.6-flash"          # fallback: any model listed by kbench.llms; record it in the report
def pick(name):
    for key in (name, name.split("/")[-1]):
        try:
            return kbench.llms[key]
        except Exception:
            pass
    raise KeyError(name)
llm = pick(JUDGE)
print(str(llm.prompt("Reply with exactly: ready", temperature=0)))

In [ ]:
PAIR_FILES = ["dpo_uf_vs_sft", "dpo_hh_vs_sft", "lora_vs_base_style", "lora_vs_base_quality", "extra_sft_vs_base"]

@kbench.task(name="sprout-pairwise-judge")
def sprout_judge(llm):
    # llm.prompt() inside a task keeps ONE growing conversation (run 1 reached 538k input
    # tokens, so later batches saw every earlier batch). fork() gives each call a throwaway
    # copy of the still-empty base chat, so every judgment is independent.
    def ask(prompt):
        with kbench.chats.fork("sprout-judge-call"):
            return str(llm.prompt(prompt, temperature=0))
    first = ask("Reply with exactly: one")
    probe = ask("What exact word did I ask you to reply with in my previous message? If there was no previous message, reply: none")
    print("isolation probe ->", probe.strip()[:80])
    assert "one" not in probe.lower().replace("none", ""), "judge still sees history; do not trust results"
    for name in PAIR_FILES:
        src = E / f"{name}_pairs.jsonl"
        if src.exists():
            print(f"\n===== {name} =====")
            judge.run(src, E / f"{name}_judged.jsonl", ask, JUDGE,
                      batch_size=5, sleep_seconds=5, max_retries=3, reuse=True)

sprout_judge.run(llm=llm)

If the task wrapper is unavailable in your notebook type, run the same loop directly:

In [ ]:
# Fallback (only if the cell above errored before judging):
# ask = lambda p: str(llm.prompt(p, temperature=0))
# for name in PAIR_FILES:
#     judge.run(E / f"{name}_pairs.jsonl", E / f"{name}_judged.jsonl", ask, JUDGE, 5, 5, 3, True)

In [ ]:
import pandas as pd
rows = []
for s in sorted(E.glob("*_judged.summary.json")):
    d = json.loads(s.read_text())
    rows.append({"comparison": s.name.replace("_judged.summary.json", ""), **{k: v for k, v in d.items() if k != "by_category"}})
table = pd.DataFrame(rows); table.to_csv(E / "summary_table.csv", index=False); table

In [ ]:
# Per-category breakdown for the HH safety set (where does DPO help or hurt?)
d = json.loads((E / "dpo_hh_vs_sft_judged.summary.json").read_text())
pd.DataFrame(d["by_category"]).T

In [ ]:
import shutil; shutil.make_archive(str(work / 'judge_outputs'), 'zip', E); print(sorted(p.name for p in E.iterdir()))